In [ ]:
import matplotlib
%matplotlib inline   # ← this is all you need in VS Code
import matplotlib.pyplot as plt

In [ ]:
!pip install tensorflow pandas matplotlib scikit-learn seaborn -q

import os, sys
sys.path.insert(0, '../')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Exploratory Data Analysis

In [ ]:
# Load annotations
df = pd.read_csv('../data/train_solution_bounding_boxes (1).csv')
df.columns = [c.strip().lower() for c in df.columns]
print('Shape:', df.shape)
df.head(10)

In [ ]:
# Stats
print('Unique images:', df['image'].nunique())
cars_per_img = df.groupby('image').size()
print('Cars per image — mean:', cars_per_img.mean().round(2),
      '  max:', cars_per_img.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cars_per_img.hist(ax=axes[0], bins=15, color='#2A9D8F', edgecolor='white')
axes[0].set_title('Cars per Image Distribution')
axes[0].set_xlabel('# cars'); axes[0].set_ylabel('# images')

# BBox area distribution
df['area'] = (df['xmax']-df['xmin']) * (df['ymax']-df['ymin'])
df['area'].hist(ax=axes[1], bins=30, color='#E63946', edgecolor='white')
axes[1].set_title('Bounding Box Area Distribution')
axes[1].set_xlabel('Area (px²)')
plt.tight_layout(); 
plt.savefig(save_path)
plt.tight_layout()

In [ ]:
# Visualise sample images with ground-truth boxes
import matplotlib.patches as patches
from tensorflow.keras.preprocessing.image import load_img, img_to_array

TRAIN_DIR = '../data/training_images'
sample_imgs = df['image'].unique()[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_name in zip(axes.flat, sample_imgs):
    img = img_to_array(load_img(os.path.join(TRAIN_DIR, img_name))).astype(np.uint8)
    ax.imshow(img)
    for _, row in df[df['image']==img_name].iterrows():
        rect = patches.Rectangle(
            (row['xmin'], row['ymin']),
            row['xmax']-row['xmin'], row['ymax']-row['ymin'],
            linewidth=2, edgecolor='#00FF88', facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title(img_name, fontsize=8); ax.axis('off')

plt.suptitle('Sample Training Images with Ground-Truth Boxes', fontsize=13, fontweight='bold')
plt.tight_layout(); 
plt.savefig(save_path)
plt.tight_layout()

## 3. Build ResNet50 Model

In [ ]:
# Import our training module
from src.train import build_resnet50_detector, IoUMetric

model = build_resnet50_detector(freeze_base=True)
model.summary()

# Visualise architecture layers
total     = sum(1 for l in model.layers)
trainable = sum(1 for l in model.layers if l.trainable)
print(f'\nTotal layers: {total}  |  Trainable: {trainable}')

## 4. Load Data & Train

In [ ]:
from src.train import CarDetectionDataset, run_experiment
from sklearn.model_selection import train_test_split

dataset = CarDetectionDataset('../data/training_images',
                               '../data/train_solution_bounding_boxes (1).csv')
X, y = dataset.build_arrays()
print('X:', X.shape, '  y:', y.shape)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', len(X_train), '  Val:', len(X_val))

In [ ]:
# Run comparison (set epochs higher for production)
results = run_experiment(X_train, y_train, X_val, y_val, epochs=15)

## 5. Results & Visualisation

In [ ]:
from src.train import plot_optimizer_comparison, print_summary_table, plot_predictions

plot_optimizer_comparison(results)
best_opt, best_model = print_summary_table(results)
plot_predictions(best_model, X_val, y_val)

## 6. Fine-Tune (Unfreeze ResNet backbone)

In [ ]:
# Unfreeze top layers of ResNet50 for fine-tuning
print('Fine-tuning best model …')
for layer in best_model.layers[-30:]:
    layer.trainable = True

best_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # Lower LR for fine-tuning
    loss='mse',
    metrics=[IoUMetric(), 'mae']
)

ft_history = best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10, batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=4,
               restore_best_weights=True, monitor='val_iou', mode='max')]
)

In [ ]:
# Final evaluation
loss, iou, mae = best_model.evaluate(X_val, y_val, verbose=0)
print(f'\n📊 Final Metrics on Validation Set')
print(f'   MSE Loss : {loss:.4f}')
print(f'   IoU      : {iou:.4f}  ({iou*100:.1f}%)')
print(f'   MAE      : {mae:.4f}')

# Save
os.makedirs('../outputs', exist_ok=True)
best_model.save('../outputs/final_model.keras')
print('\n✅ Model saved → outputs/final_model.keras')